# Hill Climbing

Implementación del algoritmo **Hill Climbing** usando las clases `Problem` y `Node` trabajadas previamente.

La idea es conservar únicamente el nodo actual, expandir sus vecinos, elegir el que tenga la menor heurística `h` y detenerse cuando se llegue a la meta o cuando ningún vecino mejore al estado actual.

## 1. Clase `Problem`

Se conserva la estructura utilizada en el notebook de A*.

In [1]:
# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial  # Estado inicial
        self.goal = goal        # Meta

    def actions(self, state):
        raise NotImplementedError

    # Función de transición
    def result(self, state, action):
        raise NotImplementedError

    # Prueba de meta
    def is_goal(self, state):
        return self.goal == state

    def action_cost(self, state1, action, state2):
        return 1

    def h(self, state):
        return 0

## 2. Problema sobre un grafo

Se utiliza la misma estructura del problema del mapa de Rumania. La heurística será la distancia en línea recta hacia Bucarest.

In [2]:
class GraphAStartProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph

    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    # Función de transición
    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

    def h(self, state):
        return straight_line_distance[state]

## 3. Clase `Node`

`expand(problem)` genera los vecinos del nodo actual y `child_node(...)` conserva el costo acumulado para poder consultar la ruta obtenida.

In [3]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

## 4. Mapa de Rumania

In [4]:
romania = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Dobreta': 75},
    'Dobreta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Dobreta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucarest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucarest': 101},
    'Bucarest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucarest': 90},
    'Urziceni': {'Bucarest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87},
}

## 5. Heurística: distancia en línea recta a Bucarest

In [5]:
straight_line_distance = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 178,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 98,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374,
}

## 6. Implementación de Hill Climbing

Correspondencia con los `TODO` de la presentación:

1. Verificar si `current` es la meta.
2. Generar vecinos con `current.expand(problem)`.
3. Elegir el vecino con **menor** valor de `h`.
4. Si el mejor vecino no mejora al actual, detenerse porque se alcanzó una cima local o meseta.

No se utiliza una frontera, conjunto de explorados ni backtracking.

In [6]:
def hill_climbing(problem):
    """Búsqueda local informada mediante descenso de h.

    Retorna:
        (current, True)  -> si alcanza la meta.
        (current, False) -> si se detiene porque no existe mejora.
    """
    current = Node(problem.initial)

    while True:
        # TODO 1: ¿current es la meta?
        if problem.is_goal(current.state):
            return current, True

        # TODO 2: generar los vecinos del estado actual
        neighbors = current.expand(problem)

        # Si no existen vecinos, no hay forma de continuar
        if not neighbors:
            return current, False

        # TODO 3: seleccionar el vecino con MENOR heurística h
        best = min(neighbors, key=lambda node: problem.h(node.state))

        # TODO 4: si el mejor vecino no mejora al actual,
        # se alcanzó una cima local / meseta
        if problem.h(best.state) >= problem.h(current.state):
            return current, False

        # Subir al mejor vecino y repetir
        current = best

## 7. Prueba: Arad → Bucarest

La traza esperada por Hill Climbing es:

**Arad → Sibiu → Fagaras → Bucarest**

In [7]:
problem = GraphAStartProblem("Arad", "Bucarest", romania)

node, success = hill_climbing(problem)

print("¿Se alcanzó la meta?:", success)
print("Ruta encontrada:", " -> ".join(node.path()))
print("Costo total:", node.path_cost, "km")
print("h del estado final:", problem.h(node.state))

¿Se alcanzó la meta?: True
Ruta encontrada: Arad -> Sibiu -> Fagaras -> Bucarest
Costo total: 450 km
h del estado final: 0


## 8. Traza paso a paso

Esta celda permite observar qué vecinos evalúa el algoritmo y cuál selecciona en cada iteración.

In [8]:
def hill_climbing_trace(problem):
    current = Node(problem.initial)
    step = 0

    while True:
        print(f"Paso {step}: {current.state} (h={problem.h(current.state)})")

        if problem.is_goal(current.state):
            print("Meta alcanzada.")
            return current, True

        neighbors = current.expand(problem)

        if not neighbors:
            print("El nodo actual no tiene vecinos.")
            return current, False

        print(
            "Vecinos:",
            ", ".join(
                f"{node.state} (h={problem.h(node.state)})"
                for node in neighbors
            )
        )

        best = min(neighbors, key=lambda node: problem.h(node.state))
        print(f"Mejor vecino: {best.state} (h={problem.h(best.state)})")

        if problem.h(best.state) >= problem.h(current.state):
            print("No existe mejora. Hill Climbing se detiene.")
            return current, False

        print(f"Movimiento: {current.state} -> {best.state}\n")
        current = best
        step += 1


trace_node, trace_success = hill_climbing_trace(problem)

Paso 0: Arad (h=366)
Vecinos: Zerind (h=374), Sibiu (h=253), Timisoara (h=329)
Mejor vecino: Sibiu (h=253)
Movimiento: Arad -> Sibiu

Paso 1: Sibiu (h=253)
Vecinos: Arad (h=366), Oradea (h=380), Fagaras (h=178), Rimnicu Vilcea (h=193)
Mejor vecino: Fagaras (h=178)
Movimiento: Sibiu -> Fagaras

Paso 2: Fagaras (h=178)
Vecinos: Sibiu (h=253), Bucarest (h=0)
Mejor vecino: Bucarest (h=0)
Movimiento: Fagaras -> Bucarest

Paso 3: Bucarest (h=0)
Meta alcanzada.


## Conclusión

Hill Climbing toma una decisión local en cada paso usando únicamente la heurística. Para el problema de Rumania llega a Bucarest desde Arad por **Arad → Sibiu → Fagaras → Bucarest**, con un costo de **450 km**. La estrategia puede detenerse antes de encontrar la meta si ningún vecino mejora el valor heurístico del estado actual.